In [1]:
# ----------------------------
# BLOQUE 1: Importar librerías
# ----------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import numpy as np


In [2]:
# ----------------------------
# BLOQUE 2: Crear sesión Spark
# ----------------------------
spark = SparkSession.builder.appName("PSI").getOrCreate()


In [3]:
# ----------------------------
# BLOQUE 3: Leer datos
# ----------------------------
merge_pyspark = spark.read.parquet("merge_pyspark")

In [4]:
# ----------------------------
# BLOQUE 4: Dividir conjuntos
# ----------------------------
# Conjunto de referencia (2018-09-20 a 2019-12-31)
conjunto1 = merge_pyspark.filter((col("Fecha") >= "2018-09-20") & (col("Fecha") <= "2019-12-31"))

# Conjunto a comparar (2020-01-01 a 2020-09-22)
conjunto2 = merge_pyspark.filter((col("Fecha") >= "2020-01-01") & (col("Fecha") <= "2020-09-22"))


In [5]:
# ----------------------------
# BLOQUE 5: Función para calcular PSI en PySpark
# ----------------------------
def calculate_psi_spark(df_expected, df_actual, column, buckets=10):
    """
    Calcula el Population Stability Index (PSI) sin usar Pandas
    df_expected: DataFrame de referencia
    df_actual: DataFrame a comparar
    column: nombre de la columna a analizar
    buckets: número de bins
    """
    # 1. Calcular percentiles aproximados del conjunto de referencia
    quantiles = df_expected.approxQuantile(column, [i/buckets for i in range(buckets+1)], 0.01)
    
    # 2. Inicializar PSI
    psi_value = 0.0
    
    # 3. Calcular PSI para cada bin
    for i in range(buckets):
        lower = quantiles[i]
        upper = quantiles[i+1]
        
        # Contar valores en cada bin
        count_expected = df_expected.filter((col(column) >= lower) & (col(column) < upper)).count()
        count_actual   = df_actual.filter((col(column) >= lower) & (col(column) < upper)).count()
        
        # Convertir a proporciones
        pct_expected = max(count_expected / df_expected.count(), 1e-6)
        pct_actual   = max(count_actual / df_actual.count(), 1e-6)
        
        # Calcular contribución al PSI
        psi_value += (pct_expected - pct_actual) * np.log(pct_expected / pct_actual)
    
    return psi_value


In [7]:
# ----------------------------
# BLOQUE 6: Calcular PSI para cada columna
# ----------------------------
psi_price  = calculate_psi_spark(conjunto1, conjunto2, "price")
psi_fn     = calculate_psi_spark(conjunto1, conjunto2, "FN")
psi_active = calculate_psi_spark(conjunto1, conjunto2, "Active")
psi_age    = calculate_psi_spark(conjunto1, conjunto2, "age")

In [8]:
# ----------------------------
# BLOQUE 7: Crear tabla con resultados (corregido)
# ----------------------------
resultados = [
    ("price", float(psi_price)),
    ("FN", float(psi_fn)),
    ("Active", float(psi_active)),
    ("age", float(psi_age))
]

# Crear DataFrame Spark sin errores de tipo
tabla_psi = spark.createDataFrame(resultados, ["Variable", "PSI"])

# Mostrar resultados
tabla_psi.show(truncate=False)

+--------+--------------------+
|Variable|PSI                 |
+--------+--------------------+
|price   |0.009615316617819275|
|FN      |0.0                 |
|Active  |0.0                 |
|age     |0.020416304850408684|
+--------+--------------------+

